In [ ]:
import os
import argparse
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

def load_vectorstore(persist_dir: str, embedding_model: str):
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
    vectordb = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
    print(f"Loaded vector store from {persist_dir}")
    return vectordb

def load_llm(model_name: str, quantize_4bit: bool = True):
    """Load a HuggingFace causal LM with optional 4-bit quantisation."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_kwargs = {
        "torch_dtype": torch.float16,
        "device_map": "auto",
    }
    if quantize_4bit:
        model_kwargs["load_in_4bit"] = True
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.1,
        repetition_penalty=1.1,
        do_sample=False
    )
    return HuggingFacePipeline(pipeline=pipe)

def main():
    parser = argparse.ArgumentParser(description="Query the medication RAG system.")
    parser.add_argument("--persist_dir", type=str, default="./med_chroma", help="Vector store directory")
    parser.add_argument("--embedding_model", type=str, default="BAAI/bge-small-en-v1.5", help="Embedding model used during ingestion")
    parser.add_argument("--llm_model", type=str, default="meta-llama/Meta-Llama-3-8B-Instruct", help="HuggingFace LLM")
    parser.add_argument("--no_quantize", action="store_true", help="Disable 4-bit quantisation (uses more GPU memory)")
    parser.add_argument("--top_k", type=int, default=5, help="Number of retrieved chunks")
    args = parser.parse_args()

    # Load vector store
    vectordb = load_vectorstore(args.persist_dir, args.embedding_model)

    # Load LLM
    print("Loading LLM (this may take a few minutes)...")
    llm = load_llm(args.llm_model, quantize_4bit=not args.no_quantize)

    # Create QA chain
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectordb.as_retriever(search_kwargs={"k": args.top_k}),
        return_source_documents=True,
        verbose=False
    )

    print("\nMedication RAG is ready. Type your question (or 'quit' to exit).")
    while True:
        query = input("\nQuestion: ")
        if query.lower() in ("quit", "exit"):
            break
        result = qa.invoke({"query": query})
        print("\nAnswer:", result["result"])
        print("\nSources:")
        for doc in result["source_documents"]:
            src = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", "")
            print(f"- {src}" + (f" (p. {page})" if page else ""))

if __name__ == "__main__":
    main()